# kline_de_pre — Kaggle 一次训练

在 Kaggle 上：**clone 代码 → 从 OKX 拉取能拿到的最多 BTC 1m 历史 → 从零训练两个模型**（CNN_Transformer + UD）。

> ⚠️ 先在右侧 **Settings → Accelerator 选 GPU**，并把 **Internet 打开**（要访问 OKX 与 GitHub）。

产出：`transformer_dct_v1.pth`（模型1）、`diffusion_delta_v1.pth`（模型2），会复制到 `/kaggle/working/` 供下载。

In [ ]:
# 1) 拉代码 + 依赖 + GPU 自检
import os
if not os.path.exists('kline-de-pre-model'):
    !git clone --depth 1 https://github.com/lin7c/kline-de-pre-model.git
%cd kline-de-pre-model
!pip -q install 'requests>=2.32' 'pandas>=2.2' 'scipy>=1.13' 'scikit-learn>=1.5'
# Kaggle 当前默认 torch 丢弃了 P100(sm_60) 内核；装一个兼容 P100+T4 的 torch(cu121)
!pip -q install torch==2.4.1 --index-url https://download.pytorch.org/whl/cu121
import torch
print('torch', torch.__version__, '| CUDA avail:', torch.cuda.is_available())
try:
    _ = (torch.zeros(8, device='cuda') + 1).sum().item()   # 真跑一下确认内核可用
    print('✅ GPU 可用:', torch.cuda.get_device_name(0))
except Exception as e:
    os.environ['CUDA_VISIBLE_DEVICES'] = ''                 # 子进程(train.py)会继承 -> 回退 CPU
    print('⚠️ GPU 内核不兼容，本次回退 CPU:', str(e)[:150])

In [ ]:
# 2) 从 OKX 拉取历史并构建数据集（正式训练：拉满，OKX 1m 可回溯约 2 年 ~100 万根）
MAX_BARS = 'all'
!python getdata_btc.py {MAX_BARS} BTC-USDT

In [ ]:
# 3) 训练流水线（预处理 → 训模型1 → 推理 → 预处理 → 训模型2）
#   正式训练：用默认轮次(10000)+早停(patience=100)；GASF 按 batch 现算；时序切分。
import subprocess, sys, os, time
steps = [
    ('CNN/makedata.py',             '数据预处理(原始窗口)'),
    ('CNN_Transformer/makedata.py', 'CNN_Transformer 标签(DCT)'),
    ('CNN_Transformer/train.py',    '训练【模型1】CNN(去Transformer)'),
    ('CNN_Transformer/gy.py',       '模型1 推理 → UD 输入'),
    ('UD/makedata.py',              'UD 残差标签'),
    ('UD/train.py',                 '训练【模型2】UD Diffusion'),
]
for script, desc in steps:
    d = os.path.dirname(script) or '.'
    print(f'\n===== {desc} =====', flush=True)
    t0 = time.time()
    subprocess.run([sys.executable, os.path.basename(script)], cwd=d, check=True)
    print(f'[{desc}] 用时 {(time.time()-t0)/60:.1f} 分钟', flush=True)

In [ ]:
# 4) 收集产物到 /kaggle/working（在 Output 面板下载）
import shutil, os
for src in ['CNN_Transformer/transformer_dct_v1.pth', 'UD/diffusion_delta_v1.pth']:
    if os.path.exists(src):
        shutil.copy(src, '/kaggle/working/' + os.path.basename(src))
        print('✅', src, '->', os.path.getsize(src) // 1024, 'KB')
    else:
        print('❌ 未找到', src)
!ls -la /kaggle/working/*.pth